In [ ]:
import pandas as pd
import numpy as np
import matplotlib as plt

In [ ]:
election_14 = pd.read_csv("/content/constituency_wise_results_2014.csv")
election_19 = pd.read_csv("/content/constituency_wise_results_2019.csv")
state_codes=pd.read_csv("/content/dim_states_codes.csv")

In [ ]:

election_14.shape

(8355, 12)

In [ ]:
election_19.shape

(8597, 12)

In [ ]:
print(election_14.columns)
print(election_19.columns)

Index(['state', 'pc_name', 'candidate', 'sex', 'age', 'category', 'party',
       'party_symbol', 'general_votes', 'postal_votes', 'total_votes',
       'total_electors'],
      dtype='object')
Index(['state', 'pc_name', 'candidate', 'sex', 'age', 'category', 'party',
       'party_symbol', 'general_votes', 'postal_votes', 'total_votes',
       'total_electors'],
      dtype='object')


In [ ]:

election_19["pc_name"]=election_19['pc_name'].str.strip().str.lower().str.replace(" ","_").str.title().str.replace("___","_").str.replace("_-_","_").str.replace("-","_")
election_14["pc_name"]=election_14['pc_name'].str.strip().str.lower().str.replace(" ","_").str.title().str.replace("___","_").str.replace("_-_","_").str.replace("-","_")

In [ ]:
state_codes=dict(zip(state_codes['state_name'],state_codes['abbreviation']))

In [ ]:
def add_abbreviation_to_pc_name(row, state_codes_dict):
    state_name = row['state']
    pc_name = row['pc_name']
    abbreviation = state_codes_dict.get(state_name)
    if abbreviation:
        return f"{pc_name}_{abbreviation.upper()}"
    return pc_name

election_14['pc_name'] = election_14.apply(lambda row: add_abbreviation_to_pc_name(row, state_codes), axis=1)
election_19['pc_name'] = election_19.apply(lambda row: add_abbreviation_to_pc_name(row, state_codes), axis=1)



#Data Cleaning
For Data cleaning and standardization , I have done some replace and strip methods on the column of constituency names columns.

Then i explored data manually by Sorting values by state, due to which we can get to see all constituencies in a a state in one sequence,which will make it easy to recognize any repeatition of names or misspell.

## Problem Due to Andhra Pradesh – Telangana bifurcation (2014 rule)

In [ ]:
election_14['state'].sort_values()
election_19['state'].sort_values()

,state
8334,Andaman & Nicobar Islands
8330,Andaman & Nicobar Islands
8331,Andaman & Nicobar Islands
8332,Andaman & Nicobar Islands
8333,Andaman & Nicobar Islands
...,...
7317,West Bengal
7318,West Bengal
7319,West Bengal
7306,West Bengal


In [ ]:
display((election_14['state']=='Telangana').sum())
display((election_14['state']=='Andhra Pradesh').sum())
display((election_19['state']=='Telangana').sum())
display((election_19['state']=='Andhra Pradesh').sum())

np.int64(0)

np.int64(640)

np.int64(460)

np.int64(344)

In [ ]:
pc_names_14 = set(election_14['pc_name'])
pc_names_19 = set(election_19['pc_name'])

unique_in_14_not_in_19 = pc_names_14 - pc_names_19
unique_in_19_not_in_14 = pc_names_19 - pc_names_14

print("\nUnique Parliamentary Constituencies in 2014 not in 2019:")
for pc in sorted(list(unique_in_14_not_in_19)):
   if pc.endswith('_TG') or pc.endswith('_AP'):
     pc=pc
     print(pc)

print("\nUnique Parliamentary Constituencies in 2019 not in 2014:")
for pc in sorted(list(unique_in_19_not_in_14)):
  if pc.endswith('_TG') or pc.endswith('_AP'):
    print(pc)


Unique Parliamentary Constituencies in 2014 not in 2019:
Adilabad_AP
Bhongir_AP
Chelvella_AP
Hyderabad_AP
Karimnagar_AP
Khammam_AP
Mahabubabad_AP
Mahbubnagar_AP
Malkajgiri_AP
Medak_AP
Nagarkurnool_AP
Nalgonda_AP
Nizamabad_AP
Peddapalle_AP
Secundrabad_AP
Warangal_AP
Zahirabad_AP

Unique Parliamentary Constituencies in 2019 not in 2014:
Adilabad_TG
Bhongir_TG
Chevella_TG
Hyderabad_TG
Karimnagar_TG
Khammam_TG
Mahabubabad_TG
Mahbubnagar_TG
Malkajgiri_TG
Medak_TG
Nagarkurnool_TG
Nalgonda_TG
Nizamabad_TG
Peddapalle_TG
Secundrabad_TG
Warangal_TG
Zahirabad_TG


In [ ]:
unique_in_14_not_in_19 = set(unique_in_14_not_in_19)

# Create a boolean mask for rows in election_14 whose 'pc_name' needs to be updated
mask_for_update = election_14['pc_name'].isin(unique_in_14_not_in_19)

# Apply the state update and pc_name replacement only to the rows identified by the mask
election_14.loc[mask_for_update, 'state'] = 'Telangana'
election_14.loc[mask_for_update, 'pc_name'] = election_14.loc[mask_for_update, 'pc_name'].str.replace('_AP', '_TG')

In [ ]:
# Update state column based on pc_name suffix for election_14
election_14.loc[election_14['pc_name'].str.endswith('_AP'), 'state'] = 'Andhra Pradesh'
election_14.loc[election_14['pc_name'].str.endswith('_TG'), 'state'] = 'Telangana'

# Update state column based on pc_name suffix for election_19
election_19.loc[election_19['pc_name'].str.endswith('_AP'), 'state'] = 'Andhra Pradesh'
election_19.loc[election_19['pc_name'].str.endswith('_TG'), 'state'] = 'Telangana'


In [ ]:
display((election_14['state']=='Telangana').sum())
display((election_14['state']=='Andhra Pradesh').sum())
display((election_19['state']=='Telangana').sum())
display((election_19['state']=='Andhra Pradesh').sum())

np.int64(323)

np.int64(358)

np.int64(460)

np.int64(344)

In [ ]:
print('Rows from election_14 where state is Andhra Pradesh or Telangana:')
display(election_14[election_14['state'].isin(['Andhra Pradesh', 'Telangana'])])

Rows from election_14 where state is Andhra Pradesh or Telangana:


,state,pc_name,candidate,sex,age,category,party,party_symbol,general_votes,postal_votes,total_votes,total_electors
0,Telangana,Adilabad_TG,GODAM NAGESH,M,49.0,ST,TRS,Car,425762,5085,430847,1386282
1,Telangana,Adilabad_TG,NARESH,M,37.0,ST,INC,Hand,257994,1563,259557,1386282
2,Telangana,Adilabad_TG,RAMESH RATHOD,M,48.0,ST,TDP,Bicycle,182879,1319,184198,1386282
3,Telangana,Adilabad_TG,RATHOD SADASHIV,M,55.0,ST,BSP,Elephant,94363,57,94420,1386282
4,Telangana,Adilabad_TG,NETHAWATH RAMDAS,M,44.0,ST,IND,Auto- Rickshaw,41028,4,41032,1386282
...,...,...,...,...,...,...,...,...,...,...,...,...
8150,Telangana,Dadar_&_Nagar_Haveli_DN,UMEDBHAI LALLUBHAI PATEL,M,47.0,ST,BMUP,Cot,348,0,348,196597
8151,Telangana,Dadar_&_Nagar_Haveli_DN,KAMUBEN KHALPABHAI PATEL,F,44.0,ST,IND,Coconut,418,0,418,196597
8152,Telangana,Dadar_&_Nagar_Haveli_DN,PRAKASH VANSHA KHANVELKAR,M,57.0,ST,IND,Whistle,1214,0,1214,196597
8153,Telangana,Dadar_&_Nagar_Haveli_DN,BHURKUD DILIPBHAI NAVJIBHAI,M,43.0,ST,IND,Kite,1154,0,1154,196597


In [ ]:
print('Rows from election_19 where state is Andhra Pradesh or Telangana:')
display(election_19[election_19['state'].isin(['Andhra Pradesh', 'Telangana'])])

Rows from election_19 where state is Andhra Pradesh or Telangana:


,state,pc_name,candidate,sex,age,category,party,party_symbol,general_votes,postal_votes,total_votes,total_electors
0,Andhra Pradesh,Aruku_AP,KISHORE CHANDRA DEO,MALE,72.0,ST,TDP,Bicycle,336163,1938,338101,1451418
1,Andhra Pradesh,Aruku_AP,Dr. KOSURI KASI VISWANADHA VEERA VENKATA SATYA...,MALE,54.0,ST,BJP,Lotus,17578,289,17867,1451418
2,Andhra Pradesh,Aruku_AP,GODDETI. MADHAVI,FEMALE,26.0,ST,YSRCP,Ceiling Fan,557561,4629,562190,1451418
3,Andhra Pradesh,Aruku_AP,SHRUTI DEVI VYRICHERLA,FEMALE,46.0,ST,INC,Hand,17656,74,17730,1451418
4,Andhra Pradesh,Aruku_AP,GANGULAIAH VAMPURU.,MALE,49.0,ST,JnP,Glass Tumbler,42245,549,42794,1451418
...,...,...,...,...,...,...,...,...,...,...,...,...
8325,Telangana,Khammam_TG,BHANALA LAXMANA CHARY,MALE,53.0,GENERAL,IND,Sewing Machine,4704,0,4704,1513809
8326,Telangana,Khammam_TG,MUTYAM ARJUNA RAJU,MALE,30.0,GENERAL,IND,Bangles,2220,0,2220,1513809
8327,Telangana,Khammam_TG,LAXMA NAIK BANOTH,MALE,62.0,ST,IND,Air Conditioner,732,2,734,1513809
8328,Telangana,Khammam_TG,SANJEEVA RAO NAKIRIKANTI,MALE,49.0,SC,IND,SHIP,2024,0,2024,1513809


##PC LEVEL

In [ ]:
aggregation_functions_pc = {
    'general_votes': 'sum',
    'postal_votes': 'sum',
    'total_votes': 'sum',
    'total_electors': 'first',  # Assuming total_electors is consistent per pc_name
    'state': 'first' # Assuming state is consistent per pc_name
}

pc_summary_14 = election_14.groupby('pc_name').agg(aggregation_functions_pc).reset_index()

print('Aggregated data for 2014 by Parliamentary Constituency:')
display(pc_summary_14.head())
pc_summary_19 = election_19.groupby('pc_name').agg(aggregation_functions_pc).reset_index()

print('Aggregated data for 2019 by Parliamentary Constituency:')
display(pc_summary_19.head())

Aggregated data for 2014 by Parliamentary Constituency:


,pc_name,general_votes,postal_votes,total_votes,total_electors,state
0,Adilabad_TG,1037720,8119,1045839,1386282,Telangana
1,Agra_UP,1069503,897,1070400,1814739,Uttar Pradesh
2,Ahmadnagar_MH,1060551,1767,1062318,1705005,Maharashtra
3,Ahmedabad_East_GJ,980265,5260,985525,1601832,Gujarat
4,Ahmedabad_West_GJ,960475,4134,964609,1534400,Gujarat


Aggregated data for 2019 by Parliamentary Constituency:


,pc_name,general_votes,postal_votes,total_votes,total_electors,state
0,Adilabad_TG,1062895,835,1063730,1489790,Telangana
1,Agra_UP,1141455,3868,1145323,1937690,Uttar Pradesh
2,Ahmadnagar_MH,1191905,11892,1203797,1861396,Maharashtra
3,Ahmedabad_East_GJ,1109296,7071,1116367,1811851,Gujarat
4,Ahmedabad_West_GJ,991051,5973,997024,1643317,Gujarat


In [ ]:
pc_summary_14['Turnout_Percentage'] = (pc_summary_14['total_votes'] / pc_summary_14['total_electors']) *100
pc_summary_19['Turnout_Percentage'] = (pc_summary_19['total_votes'] / pc_summary_19['total_electors']) *100

In [ ]:
display(pc_summary_14.sort_values(by='Turnout_Percentage',ascending=False))
display(pc_summary_19.sort_values(by='Turnout_Percentage',ascending=False))

,pc_name,general_votes,postal_votes,total_votes,total_electors,state,Turnout_Percentage
144,Dhubri_AS,1369123,501,1369624,1550166,Assam,88.353376
341,Nagaland_NL,1038361,549,1038910,1182972,Nagaland,87.822028
462,Tamluk_WB,1335058,2626,1337684,1527273,West Bengal,87.586437
99,Bishnupur_WB,1270731,1339,1272070,1466921,West Bengal,86.717008
289,Lakshadweep_LD,43239,0,43239,49922,Lakshadweep,86.613116
...,...,...,...,...,...,...,...
381,Patna_Sahib_BR,879834,2428,882262,1946249,Bihar,45.331404
244,Kalyan_MH,823598,598,824196,1922034,Maharashtra,42.881447
66,Baramulla_JK,465979,13,465992,1190766,Jammu & Kashmir,39.133801
27,Anantnag_JK,375238,41,375279,1301143,Jammu & Kashmir,28.842256


,pc_name,general_votes,postal_votes,total_votes,total_electors,state,Turnout_Percentage
155,Dhubri_AS,1682616,2389,1685005,1858566,Assam,90.661564
108,Bishnupur_WB,1417871,2777,1420648,1627199,West Bengal,87.306347
76,Barpeta_AS,1452238,6311,1458549,1685149,Assam,86.553118
235,Jalpaiguri_WB,1497711,3211,1500922,1735464,West Bengal,86.485343
36,Arunachal_East_AR,282854,10927,293781,339788,Arunachal Pradesh,86.460087
...,...,...,...,...,...,...,...
260,Kalyan_MH,887955,2358,890313,1965676,Maharashtra,45.292968
219,Hyderabad_TG,877267,605,877872,1957931,Telangana,44.836718
68,Baramulla_JK,452114,3436,455550,1317738,Jammu & Kashmir,34.570605
487,Srinagar_JK,186366,466,186832,1294671,Jammu & Kashmir,14.430848


In [ ]:
state_14=pc_summary_14.groupby('state').agg({
    'general_votes': 'sum',
    'postal_votes': 'sum',
    'total_votes': 'sum',
    'total_electors': 'sum'}).reset_index()
state_19=pc_summary_19.groupby('state').agg({
    'general_votes': 'sum',
    'postal_votes': 'sum',
    'total_votes': 'sum',
    'total_electors': 'sum'}).reset_index()

state_14['Turnout_Percentage'] = (state_14['total_votes'] / state_14['total_electors']) *100
state_19['Turnout_Percentage'] = (state_19['total_votes'] / state_19['total_electors']) *100

In [ ]:
state_14.sort_values(by='Turnout_Percentage',ascending=False).head()
state_19.sort_values(by='Turnout_Percentage',ascending=False).head()

,state,general_votes,postal_votes,total_votes,total_electors,Turnout_Percentage
17,Lakshadweep,46877,132,47009,55189,85.178206
24,Nagaland,1002142,4215,1006357,1213777,82.911194
20,Manipur,1606408,10922,1617330,1959563,82.535239
32,Tripura,2142275,10897,2153172,2614718,82.348154
35,West Bengal,57097266,109710,57206976,70001284,81.722752


In [ ]:
highest_votes_14 = election_14.loc[election_14.groupby('pc_name')['total_votes'].idxmax()]
print("Candidate with highest total votes per pc_name in 2014:")
display(highest_votes_14[['pc_name', 'candidate', 'party', 'total_votes']].head())

highest_votes_19 = election_19.loc[election_19.groupby('pc_name')['total_votes'].idxmax()]
print("\nCandidate with highest total votes per pc_name in 2019:")
display(highest_votes_19[['pc_name', 'candidate', 'party', 'total_votes']].head())

Candidate with highest total votes per pc_name in 2014:


,pc_name,candidate,party,total_votes
0,Adilabad_TG,GODAM NAGESH,TRS,430847
6161,Agra_UP,DR. RAM SHANKAR KATHERIA,BJP,583716
4090,Ahmadnagar_MH,GANDHI DILIPKUMAR MANSUKHLAL,BJP,605185
1579,Ahmedabad_East_GJ,PARESH RAWAL,BJP,633582
1594,Ahmedabad_West_GJ,DR. KIRIT P SOLANKI,BJP,617104



Candidate with highest total votes per pc_name in 2019:


,pc_name,candidate,party,total_votes
7872,Adilabad_TG,SOYAM BAPU RAO,BJP,377374
6038,Agra_UP,Satyapal Singh Baghel,BJP,646875
3877,Ahmadnagar_MH,Dr. SUJAY RADHAKRISHNA VIKHEPATIL,BJP,704660
1289,Ahmedabad_East_GJ,Patel Hasmukhbhai Somabhai,BJP,749834
1315,Ahmedabad_West_GJ,DR. KIRIT P. SOLANKI,BJP,641622


In [ ]:
winner_indices_14 = election_14.groupby('pc_name')['total_votes'].idxmax()
election_14['is_winner'] = election_14.index.isin(winner_indices_14)

winner_indices_19 = election_19.groupby('pc_name')['total_votes'].idxmax()
election_19['is_winner'] = election_19.index.isin(winner_indices_19)

Verification

In [ ]:
print(f"Number of winners in 2014: {election_14['is_winner'].value_counts().get(True, 0)}")
print(f"Total unique constituencies in 2014: {len(pc_summary_14)}")
print(f"Verification for 2014: {election_14['is_winner'].value_counts().get(True, 0) == len(pc_summary_14)}\n")

print(f"Number of winners in 2019: {election_19['is_winner'].value_counts().get(True, 0)}")
print(f"Total unique constituencies in 2019: {len(pc_summary_19)}")
print(f"Verification for 2019: {election_19['is_winner'].value_counts().get(True, 0) == len(pc_summary_19)}")

Number of winners in 2014: 511
Total unique constituencies in 2014: 511
Verification for 2014: True

Number of winners in 2019: 543
Total unique constituencies in 2019: 543
Verification for 2019: True


In [ ]:
# Get winners for 2014
winners_14 = election_14[election_14['is_winner']][['pc_name', 'candidate', 'party']]
winners_14 = winners_14.rename(columns={'candidate': 'winner_candidate', 'party': 'winner_party'})

# Merge winner info into pc_summary_14
pc_summary_14 = pc_summary_14.merge(winners_14, on='pc_name', how='left')

# Get winners for 2019
winners_19 = election_19[election_19['is_winner']][['pc_name', 'candidate', 'party']]
winners_19 = winners_19.rename(columns={'candidate': 'winner_candidate', 'party': 'winner_party'})

# Merge winner info into pc_summary_19
pc_summary_19 = pc_summary_19.merge(winners_19, on='pc_name', how='left')

In [ ]:
# Calculate winner's vote percentage for 2014
winners_14_info = election_14[election_14['is_winner']].copy()
winners_14_info = winners_14_info.rename(columns={'total_votes': 'winner_total_votes'})
winners_14_merged = pd.merge(
    winners_14_info[['pc_name', 'candidate', 'party', 'winner_total_votes']],
    pc_summary_14[['pc_name', 'total_electors']],
    on='pc_name',
    how='left'
)
winners_14_merged['winner_vote_percentage'] = (winners_14_merged['winner_total_votes'] / winners_14_merged['total_electors']) * 100
winners_14_merged['year'] = 2014

# Calculate winner's vote percentage for 2019
winners_19_info = election_19[election_19['is_winner']].copy()
winners_19_info = winners_19_info.rename(columns={'total_votes': 'winner_total_votes'})
winners_19_merged = pd.merge(
    winners_19_info[['pc_name', 'candidate', 'party', 'winner_total_votes']],
    pc_summary_19[['pc_name', 'total_electors']],
    on='pc_name',
    how='left'
)
winners_19_merged['winner_vote_percentage'] = (winners_19_merged['winner_total_votes'] / winners_19_merged['total_electors']) * 100
winners_19_merged['year'] = 2019

# Find candidates who won in both years
common_winners_names = pd.merge(
    winners_14_merged[['candidate']],
    winners_19_merged[['candidate']],
    on='candidate',
    how='inner'
)['candidate'].unique()

# Consolidate all common winners' data into a single DataFrame
all_common_winners_data = pd.concat([
    winners_14_merged[winners_14_merged['candidate'].isin(common_winners_names)],
    winners_19_merged[winners_19_merged['candidate'].isin(common_winners_names)]
])

# Calculate average vote percentage for each common winner
average_percentages = all_common_winners_data.groupby('candidate')['winner_vote_percentage'].mean().sort_values(ascending=False)

# Get the top 5 candidates by average winning percentage
top_5_candidates_by_percentage = average_percentages.head(5).index.tolist()

print("Top 5 Candidates who won in both 2014 and 2019 by average winning percentage:\n")
for winner_name in top_5_candidates_by_percentage:
    print(f"Candidate: {winner_name}")
    combined_results = pd.concat([
        winners_14_merged[winners_14_merged['candidate'] == winner_name][['year', 'pc_name', 'party', 'winner_vote_percentage']],
        winners_19_merged[winners_19_merged['candidate'] == winner_name][['year', 'pc_name', 'party', 'winner_vote_percentage']]
    ])
    display(combined_results)
    print("\n")

Top 5 Candidates who won in both 2014 and 2019 by average winning percentage:

Candidate: RAMESWAR TELI


,year,pc_name,party,winner_vote_percentage
56,2014,Dibrugarh_AS,BJP,43.970631
39,2019,Dibrugarh_AS,BJP,50.184392




Candidate: UDAY PRATAP SINGH


,year,pc_name,party,winner_vote_percentage
210,2014,Hoshangabad_MP,BJP,42.668374
193,2019,Hoshangabad_MP,BJP,51.456884




Candidate: SHRIPAD YESSO NAIK


,year,pc_name,party,winner_vote_percentage
98,2014,North_Goa_GA,BJP,46.155234
81,2019,North_Goa_GA,BJP,43.976500




Candidate: SHOBHA KARANDLAJE


,year,pc_name,party,winner_vote_percentage
160,2014,Udupi_Chikmagalur_KA,BJP,41.892202
143,2019,Udupi_Chikmagalur_KA,BJP,47.490630




Candidate: ANANTKUMAR HEGDE


,year,pc_name,party,winner_vote_percentage
157,2014,Uttara_Kannada_KA,BJP,37.704355
140,2019,Uttara_Kannada_KA,BJP,50.497398


**Reasoning**:
The first step is to group the `election_14` DataFrame by 'party' and calculate the sum of 'total_votes' for each party, storing it in `party_votes_14`. This aligns with the first instruction of the subtask.



In [ ]:
party_votes_14 = election_14.groupby('party')['total_votes'].sum().reset_index()
print('Party-wise total votes for 2014:')
display(party_votes_14.head())

Party-wise total votes for 2014:


,party,total_votes
0,A S P,557
1,AAAP,11036817
2,AAMJP,9371
3,AAP,4380
4,ABAS,296


In [ ]:
party_votes_19 = election_19.groupby('party')['total_votes'].sum().reset_index()
print('Party-wise total votes for 2019:')
display(party_votes_19.head())

Party-wise total votes for 2019:


,party,total_votes
0,AAAP,2716629
1,AABHAP,16481
2,AACP,838
3,AAHPty,3434
4,AAM,34845


In [ ]:
total_national_votes_2014 = party_votes_14['total_votes'].sum()
total_national_votes_2019 = party_votes_19['total_votes'].sum()

print(f"Total national votes in 2014: {total_national_votes_2014}")
print(f"Total national votes in 2019: {total_national_votes_2019}")

Total national votes in 2014: 520018484
Total national votes in 2019: 614172823


In [ ]:
party_votes_14['vote_percentage'] = (party_votes_14['total_votes'] / total_national_votes_2014) * 100
print('Party-wise vote percentage for 2014:')
display(party_votes_14.head())

Party-wise vote percentage for 2014:


,party,total_votes,vote_percentage
0,A S P,557,0.000107
1,AAAP,11036817,2.122389
2,AAMJP,9371,0.001802
3,AAP,4380,0.000842
4,ABAS,296,0.000057


In [ ]:
party_votes_19['vote_percentage'] = (party_votes_19['total_votes'] / total_national_votes_2019) * 100
print('Party-wise vote percentage for 2019:')
display(party_votes_19.head())

Party-wise vote percentage for 2019:


,party,total_votes,vote_percentage
0,AAAP,2716629,0.442323
1,AABHAP,16481,0.002683
2,AACP,838,0.000136
3,AAHPty,3434,0.000559
4,AAM,34845,0.005673


In [ ]:
party_votes_14['year'] = 2014
party_votes_19['year'] = 2019

combined_party_votes = pd.concat([party_votes_14, party_votes_19])

consolidated_party_shares = combined_party_votes.groupby('party')['total_votes'].sum().reset_index()

total_national_votes_combined = consolidated_party_shares['total_votes'].sum()
consolidated_party_shares['vote_percentage_combined'] = (consolidated_party_shares['total_votes'] / total_national_votes_combined) * 100

print('Consolidated party vote shares across 2014 and 2019:')
display(consolidated_party_shares.sort_values(by='vote_percentage_combined', ascending=False).head())

Consolidated party vote shares across 2014 and 2019:


,party,total_votes,vote_percentage_combined
168,BJP,390124640,34.396723
329,INC,216136963,19.056482
61,AITC,46144991,4.068537
225,BSP,44680534,3.939418
771,SP,34279774,3.022398


In [ ]:
top_two_parties = consolidated_party_shares.sort_values(by='vote_percentage_combined', ascending=False).head(2)
print('The two major national parties by combined vote share are:')
display(top_two_parties)

The two major national parties by combined vote share are:


,party,total_votes,vote_percentage_combined
168,BJP,390124640,34.396723
329,INC,216136963,19.056482


In [ ]:
def get_winner_runner_up_and_margin(df, year):
    results = []
    for pc_name, group in df.groupby('pc_name'):
        # Sort candidates by total_votes in descending order
        sorted_group = group.sort_values(by='total_votes', ascending=False)

        if len(sorted_group) >= 2:
            winner = sorted_group.iloc[0]
            runner_up = sorted_group.iloc[1]

            winning_margin = winner['total_votes'] - runner_up['total_votes']

            results.append({
                'pc_name': pc_name,
                'winner_candidate': winner['candidate'],
                'winner_party': winner['party'],
                'runner_up_candidate': runner_up['candidate'],
                'runner_up_party': runner_up['party'],
                'winning_margin': winning_margin,
                'year': year
            })
        elif len(sorted_group) == 1:
            # Handle cases with only one candidate (winner, no runner-up for margin calculation)
            winner = sorted_group.iloc[0]
            results.append({
                'pc_name': pc_name,
                'winner_candidate': winner['candidate'],
                'winner_party': winner['party'],
                'runner_up_candidate': None, # No runner-up
                'runner_up_party': None,
                'winning_margin': 0, # Or np.nan, depending on desired handling of no runner-up
                'year': year
            })

    return pd.DataFrame(results)

# Process 2014 election data
winning_margins_14 = get_winner_runner_up_and_margin(election_14, 2014)

print('Top 5 candidates with the largest winning margins in 2014:')
display(winning_margins_14.sort_values(by='winning_margin', ascending=False).head(5))

Top 5 candidates with the largest winning margins in 2014:


,pc_name,winner_candidate,winner_party,runner_up_candidate,runner_up_party,winning_margin,year
493,Vadodara_GJ,NARENDRA MODI,BJP,MISTRI MADHUSUDAN DEVRAM,INC,570128,2014
176,Ghaziabad_UP,VIJAY KUMAR SINGH,BJP,RAJ BABBAR,INC,567260,2014
357,Navsari_GJ,C. R. PATIL,BJP,MAKSUD MIRZA,INC,558116,2014
214,Jaipur_RJ,RAMCHARAN BOHARA,BJP,DR. MAHESH JOSHI,INC,539345,2014
460,Surat_GJ,DARSHANA VIKRAM JARDOSH,BJP,DESAI NAISHADHBHAI BHUPATBHAI,INC,533190,2014


In [ ]:
winning_margins_19 = get_winner_runner_up_and_margin(election_19, 2019)

print('Top 5 candidates with the largest winning margins in 2019:')
display(winning_margins_19.sort_values(by='winning_margin', ascending=False).head(5))

Top 5 candidates with the largest winning margins in 2019:


,pc_name,winner_candidate,winner_party,runner_up_candidate,runner_up_party,winning_margin,year
382,Navsari_GJ,C. R. Patil,BJP,PATEL DHARMESHBHAI BHIMBHAI,INC,689668,2019
274,Karnal_HR,Sanjay Bhatia,BJP,Kuldip Sharma,INC,656142,2019
172,Faridabad_HR,KRISHAN PAL,BJP,AVTAR SINGH BHADANA,INC,638239,2019
95,Bhilwara_RJ,SUBHASH CHANDRA BAHERIA,BJP,RAM PAL SHARMA,INC,612000,2019
525,Vadodara_GJ,RANJANBEN BHATT,BJP,PRASHANT PATEL (TIKO),INC,589177,2019


In [ ]:
party_vote_comparison = pd.merge(
    party_votes_14[['party', 'vote_percentage', 'year']],
    party_votes_19[['party', 'vote_percentage', 'year']],
    on='party',
    how='outer',
    suffixes=('_2014', '_2019')
)

# Fill NaN values with 0 for vote_percentage if a party didn't exist or participate in a given year
party_vote_comparison['vote_percentage_2014'] = party_vote_comparison['vote_percentage_2014'].fillna(0)
party_vote_comparison['vote_percentage_2019'] = party_vote_comparison['vote_percentage_2019'].fillna(0)

print('Party vote comparison DataFrame:')
display(party_vote_comparison.head())

Party vote comparison DataFrame:


,party,vote_percentage_2014,year_2014,vote_percentage_2019,year_2019
0,A S P,0.000107,2014.0,0.000000,NaN
1,AAAP,2.122389,2014.0,0.442323,2019.0
2,AABHAP,0.000000,NaN,0.002683,2019.0
3,AACP,0.000000,NaN,0.000136,2019.0
4,AAHPty,0.000000,NaN,0.000559,2019.0


In [ ]:
party_vote_comparison['vote_percentage_change'] = party_vote_comparison['vote_percentage_2019'] - party_vote_comparison['vote_percentage_2014']

print('Party vote comparison DataFrame with vote percentage change:')
display(party_vote_comparison.head())

Party vote comparison DataFrame with vote percentage change:


,party,vote_percentage_2014,year_2014,vote_percentage_2019,year_2019,vote_percentage_change
0,A S P,0.000107,2014.0,0.000000,NaN,-0.000107
1,AAAP,2.122389,2014.0,0.442323,2019.0,-1.680066
2,AABHAP,0.000000,NaN,0.002683,2019.0,0.002683
3,AACP,0.000000,NaN,0.000136,2019.0,0.000136
4,AAHPty,0.000000,NaN,0.000559,2019.0,0.000559


In [ ]:
print('Top 10 parties with the largest gains in vote percentage:')
display(party_vote_comparison.sort_values(by='vote_percentage_change', ascending=False).head(10))

Top 10 parties with the largest gains in vote percentage:


,party,vote_percentage_2014,year_2014,vote_percentage_2019,year_2019,vote_percentage_change
168,BJP,30.969622,2014.0,37.298440,2019.0,6.328817
256,CPIM,0.017085,2014.0,1.764251,2019.0,1.747167
162,BJD,0.000000,NaN,1.656540,2019.0,1.656540
329,INC,18.584291,2014.0,19.456285,2019.0,0.871994
845,VBA,0.000000,NaN,0.609529,2019.0,0.609529
270,DMK,1.852097,2014.0,2.338647,2019.0,0.486550
411,JnP,0.000000,NaN,0.311822,2019.0,0.311822
359,JD(U),1.150845,2014.0,1.453447,2019.0,0.302602
547,NTK,0.000000,NaN,0.275993,2019.0,0.275993
480,MNM,0.000000,NaN,0.262745,2019.0,0.262745


In [ ]:
print('Bottom 10 parties with the largest losses in vote percentage:')
display(party_vote_comparison.sort_values(by='vote_percentage_change', ascending=True).head(10))

Bottom 10 parties with the largest losses in vote percentage:


,party,vote_percentage_2014,year_2014,vote_percentage_2019,year_2019,vote_percentage_change
257,CPM,3.449519,2014.0,0.000000,NaN,-3.449519
34,ADMK,3.482872,2014.0,1.352607,2019.0,-2.130265
1,AAAP,2.122389,2014.0,0.442323,2019.0,-1.680066
771,SP,3.583059,2014.0,2.547688,2019.0,-1.035371
225,BSP,4.314084,2014.0,3.622189,2019.0,-0.691895
810,TDP,2.711294,2014.0,2.037756,2019.0,-0.673538
334,INLD,0.538423,2014.0,0.039119,2019.0,-0.499304
331,IND,3.055294,2014.0,2.700107,2019.0,-0.355187
637,RJD,1.430899,2014.0,1.079867,2019.0,-0.351032
513,NCP,1.660625,2014.0,1.384029,2019.0,-0.276596


In [ ]:
state_party_votes_14 = election_14.groupby(['state', 'party'])['total_votes'].sum().reset_index()
print('State-wise and Party-wise total votes for 2014:')
display(state_party_votes_14.head())

State-wise and Party-wise total votes for 2014:


,state,party,total_votes
0,Andaman & Nicobar Islands,AAAP,3737
1,Andaman & Nicobar Islands,AIFB,225
2,Andaman & Nicobar Islands,AITC,2283
3,Andaman & Nicobar Islands,BJP,90969
4,Andaman & Nicobar Islands,BSP,1139


In [ ]:
total_state_votes_14 = state_party_votes_14.groupby('state')['total_votes'].sum().reset_index()
total_state_votes_14.rename(columns={'total_votes': 'total_state_votes'}, inplace=True)

state_party_votes_14 = pd.merge(state_party_votes_14, total_state_votes_14, on='state', how='left')
state_party_votes_14['vote_percentage'] = (state_party_votes_14['total_votes'] / state_party_votes_14['total_state_votes']) * 100
state_party_votes_14['year'] = 2014

print('State-wise and Party-wise vote percentage for 2014:')
display(state_party_votes_14.head())

State-wise and Party-wise vote percentage for 2014:


,state,party,total_votes,total_state_votes,vote_percentage,year
0,Andaman & Nicobar Islands,AAAP,3737,190328,1.963453,2014
1,Andaman & Nicobar Islands,AIFB,225,190328,0.118217,2014
2,Andaman & Nicobar Islands,AITC,2283,190328,1.199508,2014
3,Andaman & Nicobar Islands,BJP,90969,190328,47.795910,2014
4,Andaman & Nicobar Islands,BSP,1139,190328,0.598441,2014


In [ ]:
state_party_votes_19 = election_19.groupby(['state', 'party'])['total_votes'].sum().reset_index()

total_state_votes_19 = state_party_votes_19.groupby('state')['total_votes'].sum().reset_index()
total_state_votes_19.rename(columns={'total_votes': 'total_state_votes'}, inplace=True)

state_party_votes_19 = pd.merge(state_party_votes_19, total_state_votes_19, on='state', how='left')
state_party_votes_19['vote_percentage'] = (state_party_votes_19['total_votes'] / state_party_votes_19['total_state_votes']) * 100
state_party_votes_19['year'] = 2019

print('State-wise and Party-wise vote percentage for 2019:')
display(state_party_votes_19)

State-wise and Party-wise vote percentage for 2019:


,state,party,total_votes,total_state_votes,vote_percentage,year
0,Andaman & Nicobar Islands,AAAP,2839,207296,1.369539,2019
1,Andaman & Nicobar Islands,AINHCP,212,207296,0.102269,2019
2,Andaman & Nicobar Islands,AITC,1721,207296,0.830214,2019
3,Andaman & Nicobar Islands,BJP,93901,207296,45.298028,2019
4,Andaman & Nicobar Islands,BSP,2486,207296,1.199251,2019
...,...,...,...,...,...,...
1381,West Bengal,SUCI(C),217376,57206976,0.379982,2019
1382,West Bengal,SWJP,1899,57206976,0.003320,2019
1383,West Bengal,UTSAP,2375,57206976,0.004152,2019
1384,West Bengal,WPOI,23035,57206976,0.040266,2019


In [ ]:
state_party_comparison = pd.merge(
    state_party_votes_14[['state', 'party', 'vote_percentage', 'year']],
    state_party_votes_19[['state', 'party', 'vote_percentage', 'year']],
    on=['state', 'party'],
    how='outer',
    suffixes=('_2014', '_2019')
)

# Fill NaN values with 0 for vote_percentage if a party didn't exist or participate in a given year
state_party_comparison['vote_percentage_2014'] = state_party_comparison['vote_percentage_2014'].fillna(0)
state_party_comparison['vote_percentage_2019'] = state_party_comparison['vote_percentage_2019'].fillna(0)

print('State-wise and Party-wise vote percentage comparison DataFrame:')
display(state_party_comparison.head())

State-wise and Party-wise vote percentage comparison DataFrame:


,state,party,vote_percentage_2014,year_2014,vote_percentage_2019,year_2019
0,Andaman & Nicobar Islands,AAAP,1.963453,2014.0,1.369539,2019.0
1,Andaman & Nicobar Islands,AIFB,0.118217,2014.0,0.000000,NaN
2,Andaman & Nicobar Islands,AINHCP,0.000000,NaN,0.102269,2019.0
3,Andaman & Nicobar Islands,AITC,1.199508,2014.0,0.830214,2019.0
4,Andaman & Nicobar Islands,BJP,47.795910,2014.0,45.298028,2019.0


In [ ]:
state_party_comparison['vote_percentage_change'] = state_party_comparison['vote_percentage_2019'] - state_party_comparison['vote_percentage_2014']

print('State-wise and Party-wise vote percentage comparison DataFrame with change:')
display(state_party_comparison.head())

State-wise and Party-wise vote percentage comparison DataFrame with change:


,state,party,vote_percentage_2014,year_2014,vote_percentage_2019,year_2019,vote_percentage_change
0,Andaman & Nicobar Islands,AAAP,1.963453,2014.0,1.369539,2019.0,-0.593913
1,Andaman & Nicobar Islands,AIFB,0.118217,2014.0,0.000000,NaN,-0.118217
2,Andaman & Nicobar Islands,AINHCP,0.000000,NaN,0.102269,2019.0,0.102269
3,Andaman & Nicobar Islands,AITC,1.199508,2014.0,0.830214,2019.0,-0.369294
4,Andaman & Nicobar Islands,BJP,47.795910,2014.0,45.298028,2019.0,-2.497882


In [ ]:
major_parties = top_two_parties['party'].tolist()

print(f"The two major national parties identified are: {major_parties}")

The two major national parties identified are: ['BJP', 'INC']


In [ ]:
for party_name in major_parties:
    print(f"\nAnalyzing vote share for: {party_name}")

    # 2a. Filter election_14 for current party and calculate total votes per pc_name
    party_14_pc_votes = election_14[election_14['party'] == party_name].groupby('pc_name')['total_votes'].sum().reset_index()
    party_14_pc_votes.rename(columns={'total_votes': 'party_total_votes'}, inplace=True)

    # 2b. Merge with pc_summary_14 to get total_electors
    party_pc_votes_14 = pd.merge(
        party_14_pc_votes,
        pc_summary_14[['pc_name', 'total_electors']],
        on='pc_name',
        how='left'
    )

    # 2c. Calculate vote_percentage_2014
    party_pc_votes_14['vote_percentage_2014'] = (party_pc_votes_14['party_total_votes'] / party_pc_votes_14['total_electors']) * 100

    # 2d. Filter election_19 for current party and calculate total votes per pc_name
    party_19_pc_votes = election_19[election_19['party'] == party_name].groupby('pc_name')['total_votes'].sum().reset_index()
    party_19_pc_votes.rename(columns={'total_votes': 'party_total_votes'}, inplace=True)

    # 2e. Merge with pc_summary_19 to get total_electors
    party_pc_votes_19 = pd.merge(
        party_19_pc_votes,
        pc_summary_19[['pc_name', 'total_electors']],
        on='pc_name',
        how='left'
    )

    # 2f. Calculate vote_percentage_2019
    party_pc_votes_19['vote_percentage_2019'] = (party_pc_votes_19['party_total_votes'] / party_pc_votes_19['total_electors']) * 100

    # 2g. Merge 2014 and 2019 vote percentages
    party_constituency_comparison = pd.merge(
        party_pc_votes_14[['pc_name', 'vote_percentage_2014']],
        party_pc_votes_19[['pc_name', 'vote_percentage_2019']],
        on='pc_name',
        how='outer'
    )
    party_constituency_comparison['vote_percentage_2014'] = party_constituency_comparison['vote_percentage_2014'].fillna(0)
    party_constituency_comparison['vote_percentage_2019'] = party_constituency_comparison['vote_percentage_2019'].fillna(0)

    # 2h. Calculate vote_percentage_change
    party_constituency_comparison['vote_percentage_change'] = party_constituency_comparison['vote_percentage_2019'] - party_constituency_comparison['vote_percentage_2014']

    # 2i. Sort and display top 5 constituencies with largest gains
    print(f"Top 5 constituencies where {party_name} gained the most vote share (2019 vs 2014):")
    display(party_constituency_comparison.sort_values(by='vote_percentage_change', ascending=False).head(5))



Analyzing vote share for: BJP
Top 5 constituencies where BJP gained the most vote share (2019 vs 2014):


,pc_name,vote_percentage_2014,vote_percentage_2019,vote_percentage_change
154,Durg_CT,0.0,43.776095,43.776095
402,Sarguja_CT,0.0,40.097593,40.097593
376,Raipur_CT,0.0,39.678312,39.678312
418,Sirsa_HR,0.0,39.612465,39.612465
382,Rajnandgaon_CT,0.0,38.590319,38.590319



Analyzing vote share for: INC
Top 5 constituencies where INC gained the most vote share (2019 vs 2014):


,pc_name,vote_percentage_2014,vote_percentage_2019,vote_percentage_change
251,Karur_TN,2.346028,50.148059,47.802031
30,Arani_TN,2.023629,42.629127,40.605498
449,Tiruchirappalli_TN,3.715342,41.172978,37.457635
276,Krishnagiri_TN,2.817841,39.943570,37.125729
444,Thiruvallur_TN,2.581580,39.411568,36.829988


In [ ]:
for party_name in major_parties:
    print(f"\nAnalyzing vote share for: {party_name}")

    # Filter election_14 for current party and calculate total votes per pc_name
    party_14_pc_votes = election_14[election_14['party'] == party_name].groupby('pc_name')['total_votes'].sum().reset_index()
    party_14_pc_votes.rename(columns={'total_votes': 'party_total_votes'}, inplace=True)

    # Merge with pc_summary_14 to get total_electors
    party_pc_votes_14 = pd.merge(
        party_14_pc_votes,
        pc_summary_14[['pc_name', 'total_electors']],
        on='pc_name',
        how='left'
    )

    # Calculate vote_percentage_2014
    party_pc_votes_14['vote_percentage_2014'] = (party_pc_votes_14['party_total_votes'] / party_pc_votes_14['total_electors']) * 100

    # Filter election_19 for current party and calculate total votes per pc_name
    party_19_pc_votes = election_19[election_19['party'] == party_name].groupby('pc_name')['total_votes'].sum().reset_index()
    party_19_pc_votes.rename(columns={'total_votes': 'party_total_votes'}, inplace=True)

    # Merge with pc_summary_19 to get total_electors
    party_pc_votes_19 = pd.merge(
        party_19_pc_votes,
        pc_summary_19[['pc_name', 'total_electors']],
        on='pc_name',
        how='left'
    )

    # Calculate vote_percentage_2019
    party_pc_votes_19['vote_percentage_2019'] = (party_pc_votes_19['party_total_votes'] / party_pc_votes_19['total_electors']) * 100

    # Merge 2014 and 2019 vote percentages
    party_constituency_comparison = pd.merge(
        party_pc_votes_14[['pc_name', 'vote_percentage_2014']],
        party_pc_votes_19[['pc_name', 'vote_percentage_2019']],
        on='pc_name',
        how='outer'
    )
    party_constituency_comparison['vote_percentage_2014'] = party_constituency_comparison['vote_percentage_2014'].fillna(0)
    party_constituency_comparison['vote_percentage_2019'] = party_constituency_comparison['vote_percentage_2019'].fillna(0)

    # Calculate vote_percentage_change
    party_constituency_comparison['vote_percentage_change'] = party_constituency_comparison['vote_percentage_2019'] - party_constituency_comparison['vote_percentage_2014']

    # Sort and display top 5 constituencies with largest losses (ascending order)
    print(f"Top 5 constituencies where {party_name} lost the most vote share (2019 vs 2014):")
    display(party_constituency_comparison.sort_values(by='vote_percentage_change', ascending=True).head(5))


Analyzing vote share for: BJP
Top 5 constituencies where BJP lost the most vote share (2019 vs 2014):


,pc_name,vote_percentage_2014,vote_percentage_2019,vote_percentage_change
129,Dadar_&_Nagar_Haveli_DN,41.094218,0.000000,-41.094218
337,Narsapuram_AP,40.776950,0.862268,-39.914681
100,Bikaner_RJ,36.733769,0.000000,-36.733769
351,Palghar_MH,33.786480,0.000000,-33.786480
442,Tirupati_AP,34.491453,0.977004,-33.514448



Analyzing vote share for: INC
Top 5 constituencies where INC lost the most vote share (2019 vs 2014):


,pc_name,vote_percentage_2014,vote_percentage_2019,vote_percentage_change
126,Dadar_&_Nagar_Haveli_DN,37.933437,0.0,-37.933437
303,Mandya_KA,31.082718,0.0,-31.082718
314,Mizoram_MZ,29.976359,0.0,-29.976359
461,Udupi_Chikmagalur_KA,28.798870,0.0,-28.798870
192,Hatkanangle_MH,28.370959,0.0,-28.370959


## Constituency with Most NOTA Votes




In [ ]:
nota_votes_14 = election_14[election_14['party'] == 'NOTA'].groupby('pc_name')['total_votes'].sum().reset_index()
nota_votes_14.rename(columns={'total_votes': 'nota_total_votes'}, inplace=True)
print('NOTA votes per constituency for 2014:')
display(nota_votes_14.head())

NOTA votes per constituency for 2014:


,pc_name,nota_total_votes
0,Adilabad_TG,17084
1,Agra_UP,5191
2,Ahmadnagar_MH,7473
3,Ahmedabad_East_GJ,14358
4,Ahmedabad_West_GJ,16571


In [ ]:
nota_votes_19 = election_19[election_19['party'] == 'NOTA'].groupby('pc_name')['total_votes'].sum().reset_index()
nota_votes_19.rename(columns={'total_votes': 'nota_total_votes'}, inplace=True)
print('NOTA votes per constituency for 2019:')
display(nota_votes_19.head())

NOTA votes per constituency for 2019:


,pc_name,nota_total_votes
0,Adilabad_TG,13036
1,Agra_UP,5817
2,Ahmadnagar_MH,4072
3,Ahmedabad_East_GJ,9008
4,Ahmedabad_West_GJ,14719


In [ ]:
combined_nota_votes = pd.concat([nota_votes_14, nota_votes_19], ignore_index=True)
print('Combined NOTA votes across 2014 and 2019:')
display(combined_nota_votes.head())

Combined NOTA votes across 2014 and 2019:


,pc_name,nota_total_votes
0,Adilabad_TG,17084
1,Agra_UP,5191
2,Ahmadnagar_MH,7473
3,Ahmedabad_East_GJ,14358
4,Ahmedabad_West_GJ,16571


In [ ]:
total_nota_votes_by_constituency = combined_nota_votes.groupby('pc_name')['nota_total_votes'].sum().reset_index()
total_nota_votes_by_constituency.rename(columns={'nota_total_votes': 'total_nota_votes'}, inplace=True)
print('Total NOTA votes per constituency across 2014 and 2019:')
display(total_nota_votes_by_constituency.head())

Total NOTA votes per constituency across 2014 and 2019:


,pc_name,total_nota_votes
0,Adilabad_TG,30120
1,Agra_UP,11008
2,Ahmadnagar_MH,11545
3,Ahmedabad_East_GJ,23366
4,Ahmedabad_West_GJ,31290


In [ ]:
highest_nota_constituency = total_nota_votes_by_constituency.loc[total_nota_votes_by_constituency['total_nota_votes'].idxmax()]
print('Constituency with the highest NOTA votes across 2014 and 2019:')
display(highest_nota_constituency)

Constituency with the highest NOTA votes across 2014 and 2019:


,198
pc_name,Gopalganj_(Sc)_BR
total_nota_votes,69501
